# Create materialized lake views 
1. Use this notebook to create materialized lake views. 
2. Select **Run all** to run the notebook. 
3. When the notebook run is completed, return to your lakehouse and refresh your materialized lake views graph. 


In [2]:
%%pyspark
import requests
import time

api_key = "c0599ff26b1b365e955bb4be47ac042b60245a76f4a3969e8886d5189e063366"
headers = {"X-API-Key": api_key}

locations = []
page = 1
while True:
    url = (
        "https://api.openaq.org/v3/locations"
        "?bbox=-74.25,40.50,-73.70,40.92"
        "&limit=100&page="
        + str(page)
    )
    r = requests.get(url, headers=headers)
    results = r.json().get("results", [])
    if not results:
        break
    locations.extend(results)
    page += 1
    if len(results) < 100:
        break
    time.sleep(0.3)

len(locations)

StatementMeta(, 582efaff-fe1f-4693-a1f2-1b4e7f74d6dc, 4, Finished, Available, Finished, False)

62

In [3]:
%%pyspark
def get_all_sensors(locations):
    sensors = []
    for loc in locations:
        loc_id = loc["id"]
        loc_name = loc["name"]
        for sensor in loc.get("sensors", []):
            sensors.append({
                "sensor_id": sensor["id"],
                "location_id": loc_id,
                "location_name": loc_name,
                "parameter": sensor["parameter"]["name"],
                "units": sensor["parameter"]["units"],
            })
    return sensors

sensors = get_all_sensors(locations)
len(sensors)

StatementMeta(, 582efaff-fe1f-4693-a1f2-1b4e7f74d6dc, 5, Finished, Available, Finished, False)

225

In [4]:
%%pyspark
all_rows = []

for s in sensors:
    page = 1
    while True:
        url = (
            "https://api.openaq.org/v3/sensors/"
            + str(s["sensor_id"])
            + "/measurements?date_from=2025-01-01T00:00:00Z"
            + "&date_to=2025-05-16T23:59:59Z"
            + "&limit=1000&page="
            + str(page)
        )
        resp = requests.get(url, headers=headers)
        if resp.status_code != 200:
            break
        results = resp.json().get("results", [])
        if not results:
            break
        for m in results:
            all_rows.append({
                "sensor_id": s["sensor_id"],
                "location_id": s["location_id"],
                "location_name": s["location_name"],
                "parameter": s["parameter"],
                "units": s["units"],
                "value": m.get("value"),
                "datetime_utc": m.get("period", {}).get("datetimeFrom", {}).get("utc"),
                "datetime_local": m.get("period", {}).get("datetimeFrom", {}).get("local"),
            })
        page += 1
        if len(results) < 1000:
            break
        time.sleep(0.3)
    time.sleep(0.5)

len(all_rows)

StatementMeta(, 582efaff-fe1f-4693-a1f2-1b4e7f74d6dc, 6, Submitted, Running, Running, True)

In [ ]:
%%pyspark
import pandas as pd

pdf = pd.DataFrame(all_rows)
df = spark.createDataFrame(pdf)
df.write.mode("append").saveAsTable("bronze_air_quality")
df.count()

In [22]:
%%pyspark
import pandas as pd

pdf = pd.DataFrame(all_rows)
df = spark.createDataFrame(pdf)
df.write.mode("overwrite").saveAsTable("bronze_air_quality")

df.count()

StatementMeta(, 2be97449-24b8-493e-94d8-4cbc140624f5, 24, Finished, Available, Finished, False)

212958

In [1]:
%%pyspark
import requests

r = requests.get("https://api.worldbank.org/v2/country/USA/indicator/NY.GDP.MKTP.CD?format=json&per_page=100")
data = r.json()
records = data[1]

gdp_rows = []
for item in records:
    gdp_rows.append({
        "country_code": item.get("countryiso3code"),
        "country_name": item.get("country", {}).get("value"),
        "year": item.get("date"),
        "gdp_usd": item.get("value"),
    })

len(gdp_rows)

StatementMeta(, 68b367b7-ce5a-4510-a145-ed0bef441723, 3, Finished, Available, Finished, False)

66

In [2]:
%%pyspark
import pandas as pd

pdf_gdp = pd.DataFrame(gdp_rows)
df_gdp = spark.createDataFrame(pdf_gdp)
df_gdp.write.mode("overwrite").saveAsTable("bronze_gdp")
df_gdp.count()

StatementMeta(, 68b367b7-ce5a-4510-a145-ed0bef441723, 4, Finished, Available, Finished, False)

66

In [3]:
%%pyspark
import csv
import io

r = requests.get("https://data-api.ecb.europa.eu/service/data/EXR/D.USD.EUR.SP00.A?format=csvdata")
r.encoding = "utf-8"
content = r.content.decode("utf-8")

fx_rows = []
reader = csv.DictReader(io.StringIO(content))
for row in reader:
    fx_rows.append({
        "date": row.get("TIME_PERIOD"),
        "usd_eur_rate": row.get("OBS_VALUE"),
        "currency": "USD/EUR",
    })

len(fx_rows)

StatementMeta(, 68b367b7-ce5a-4510-a145-ed0bef441723, 5, Finished, Available, Finished, False)

7068

In [4]:
%%pyspark
import pandas as pd

pdf_fx = pd.DataFrame(fx_rows)
df_fx = spark.createDataFrame(pdf_fx)
df_fx.write.mode("overwrite").saveAsTable("bronze_fx_rates")
df_fx.count()

StatementMeta(, 68b367b7-ce5a-4510-a145-ed0bef441723, 6, Finished, Available, Finished, False)

7068